### Individual Planning Report

Vicky N _ Individual Planning Report _ 11 / 12 / 2025

In [188]:
library(repr)
library(tidyverse)
library(tidymodels)
source("cleanup.R")

Warning message in file(filename, "r", encoding = encoding):
“cannot open file 'cleanup.R': No such file or directory”


ERROR: Error in file(filename, "r", encoding = encoding): cannot open the connection


In [ ]:
# Loading in Players Data 
players_url <- ("https://raw.githubusercontent.com/vicky-nak/Individual-Planning-Report/refs/heads/main/Individual%20Planning%20Report/players.csv")
players <- read_csv(players_url)

**Data Description** _Breakdown of minecraft server player data_

##### Players data 

###### DATA COLLECTION 
A minecreaft server was set up by a research group at UBC to record how people play video games. Variables such as name and gender are input by the user while variables like played hours are calculated by the game. This dataset contains 196 observations and 7 variables. 

###### DATA SUMMARY

In [ ]:
# Experience summary
Pros      <- players |> filter(experience == "Pro")      |> nrow()
Amateurs  <- players |> filter(experience == "Amateur")  |> nrow()
Beginners <- players |> filter(experience == "Beginner") |> nrow() 
Veterans  <- players |> filter(experience == "Veteran")  |> nrow()
Regulars  <- players |> filter(experience == "Regular")  |> nrow()
# Subscribe summary
Subscribed    <- players |> filter(subscribe == "TRUE")  |> nrow()
Not_subcribed <- players |> filter(subscribe == "FALSE") |> nrow() 
# Played hours summary
Mean_played_hours <- players |> filter(!is.na(played_hours)) |> summarize(mean(played_hours)) |>  as.numeric()
Unplaying_players <- players |> filter(!is.na(played_hours)) |> filter(played_hours == 0) |> nrow()
# Gender summary
Males             <- players |> filter(gender == "Male")              |> nrow()
Females           <- players |> filter(gender == "Female")            |> nrow() 
Two_spirited      <- players |> filter(gender == "Two-Spirited")      |> nrow() 
Non_binary        <- players |> filter(gender == "Non-binary")        |> nrow() 
Agender           <- players |> filter(gender == "Agender")           |> nrow()
Other             <- players |> filter(gender == "Other")             |> nrow() 
Prefer_not_to_say <- players |> filter(gender == "Prefer not to say") |> nrow() 
# Age summary
Mean_age   <- players |> filter(!is.na(Age), na.rm = TRUE) |> summarize(mean(Age)) |> as.numeric()
Median_age <- players |> filter(!is.na(Age), na.rm = TRUE) |> summarize(median(Age)) |> as.numeric()

# Player Data summary
Experience_summary <- c(Pros, Amateurs, Beginners, Veterans, Regulars) |> print()
Subscribe_summary <- c(Subscribed, Not_subcribed) |> print()
Played_hours_summary <- c(Mean_played_hours, Unplaying_players) |> print()
Gender_summary <- c(Males, Females, Two_spirited, Non_binary, Agender, Other, Prefer_not_to_say) |> print()
Age_summary <- c(Mean_age, Median_age) |> print()

###### VARIABLE ANALYSIS

| Variable       | Type      | Description w/ examples                                                    | Issues / Changes               | Statistics / Observations
| -------------- | --------- | -------------------------------------------------------------------------- | ------------------------------ | --------------------------------------------------------------------
| `experience`   | character | A players self-declared level of knowledge on the game                     | character -> factor            | `Amateur = 63` `Beginner = 35` `Pro = 14` `Veteran = 48` `Regular = 36`
| `subscribe`    | logical   | Whether a player is subscribed to the newsletter `TRUE` or `FALSE`         | subscribe -> `subscribed`      | `Subscribed = 144` `Not subcribed = 52`
| `hashedEmail`  | character | Encrypted email, a collection of numbers and lower case letters            | hashedEmail -> `hashed_email`  | 
| `played_hours` | double    | Hours of game time logged as a double to one decimal place                 |                                | `Mean played hours = 5.85` `Unplaying players = 85`
| `name`         | character | A players name, may have duplicates that do not represent the same player  |                                | 
| `gender`       | character | The gender of the player                                                   | character -> factor            | `Males = 124` `Females = 37` `Two-spirited = 6` `Non-binary = 15` `Agender = 2` `Other = 1` `Prefer not to say = 11`
| `Age`          | double    | Age of player (integer)                                                           | double -> integer, Age -> `age` | `Mean age = 21.14` `Median age = 19`


###### POTENTIAL ISSUES
Aside from naming and data type changes mentioned in the variable analysis table, any N/A values would need to be delt with such as the age in the last observation. If there are any duplicates in the names then they are not useful as identifiers and we must rely on the `hashedEmail`.



**Questions**

###### RESEARCH QUESTION
I will be addressing which "kinds" of players are most likely to contribute a large amount of data so that we can target those players in our recruiting efforts. Summarizing the dataset revelaed around 43% of players never play. According to Plaicraft rules, each new player recieves 30 minutes of time they are allowed to play alone without interacting with others. Making 30 minutes a substantial amount of data for the research. Therefore, I want to find out: **_Does a player's experience, age, and gender predict where a player would log more than half an hour on the game?_** This question requires played_hours to be mutated into a logical variable of players that played more than 0.5 hours. Given I would be predicting a categorical (logical) variable, I would most likely use knn classification.

**Exploratory Data Analysis and Visualization**

In [ ]:
# Loading in the players dataset was done at the beginning of the file
slice(players, 1:3)

###### TIDY DATA
The data for players is already tidy as it meets the three criteria for tidy data. Each row did represent a single obervation or player in this case. Each column was a single variable, and while some data types could be changes, the columns were tidy nonetheless. Lastly, the values in each cell were singular and representatitve of one value only.

###### MEAN VALUES

In [ ]:
means_of_quantitative_variables <- players |>
    select(played_hours, Age) |>
    summarize(mean_played_hours = mean(played_hours), mean_age = mean(Age, na.rm = TRUE))
means_of_quantitative_variables

###### EXPLORATORY VISUALIZATION

In [ ]:
# A scatterplot showing the relationship between age and hours players
options(repr.plot.width = 15, repr.plot.height = 6)

age_vs_played_hours <- players |>
    ggplot(aes(x = Age, y = played_hours)) +
           geom_point(alpha = 0.4) +
           labs(x = "Age of Player (years)", y = "Hours Played (hours)") +
           theme(text = element_text(size = 18)) +
           ggtitle("Does a player's age correlate with the time they play?")
age_vs_played_hours

###### Does a player's age correlate with the time they play?  |  age_vs_played_hours analysis
This graph does not tell us much given the extremes between the hours players played. We can tell that ges 10 to 25 are more likely to log higher hours than ages 30+. Given the outliers, as our research question states, categorizing the hours played may be very helpful. Removing players that did not play at all may also help us gain more insights about the data. Using a facet grid with our scatterplot may allow us to observe more insights. 

In [ ]:
# A histogram showing the chosen explanatory variables
options(repr.plot.width = 15, repr.plot.height = 8)
players_with_logged_hours <- players |>
    filter(played_hours > 0, played_hours < 60)
played_hours_grid <- players_with_logged_hours |>
   ggplot(aes(x = Age, y = played_hours)) +
           geom_point(alpha = 0.4) +
           facet_grid(rows = vars(experience), cols = vars(gender)) +
           labs(x = "Age of Player (years)", y = "Hours Played (hours)") +
           theme(text = element_text(size = 18)) +
           ggtitle("Experience, age, and gender's effect on Hours Played")
played_hours_grid

###### Experience, age, and gender's effect on Hours Played  |  played_hours_grid analysis
This facet grid tells us much more and gives an overlooking view at the data. From it we can garner insights such as there is indeed a range of ages more likely to spend more hours playing.
The data will need to be standardized and scaled in multiple areas as it is evident males make up a significant portion of the players and many players are amateurs. Therefore it is not okay to compare between these variables when they have many caterigories overrepresented in the data.

**Methods and Plan**

In order to answer whether a player's experience, age, and gender predict where a player would log more than half an hour on the game I belive a knn classification on the players.csv dataset would be most appropriate. This is becuase the sessions.csv dataset covers individual player sessions and how long they lasted which does not contain any player specific information. As for the knn clasificiation, it is the appropriate method because we would be classifying a qualitative variable. 

It is important to note in it's current state, our variable of interest in not the right data-type. I would first have to manipulate the `played_hours` variable to make it categorical or logical, representing various levels of hours played.

Knn clasification is sensitive to unstandardized data therefore as part of preprocessing, I would `step_scale` and `step-center` the data. To do a proper classifications, I will split the data into training and testing set with a standared 0.75 `prop` split. I would do the spliting at the begining of the processing, and then use 5-fold-cross-validation on the training set to select an optimal k. 

In [ ]:
source("cleanup.R")